In [ ]:
# Imports
import pandas as pd
import numpy as np

In [ ]:
# USER CONFIGURATION
# Change these paths

# Source data
ORIGINAL_DATA_PATH = "source_data.csv"
# Deduplication data [source_id, found_lipas_id]
DEDUP_PATH = "deduplication_data.csv"
# Mapping table 
MAPPING_PATH = "mapping_table.csv"

original_df = pd.read_csv(ORIGINAL_DATA_PATH)
dedup_df = pd.read_csv(DEDUP_PATH)
mapping_df = pd.read_csv(MAPPING_PATH, sep = ";")

print("Original rows:", original_df.shape)
print("Dedup rows:", dedup_df.shape)
print("Mapping rows:", mapping_df.shape)

display(original_df.head())
display(dedup_df.head())
display(mapping_df.head())

In [ ]:
# USER CONFIGURATION
# Add here the column names used in this mapping workflow

# Original source data columns
SOURCE_ID_COL = "Stable unique identifier in the original data"
SOURCE_TYPE_COL = "Original category/type"       
SOURCE_NAME_COL = "Original object name"        

# Deduplication result columns
DEDUP_SOURCE_ID_COL = "Same identifier as source_id, coming from dedup result" 
DEDUP_LIPAS_ID_COL = "Existing Lipas id > 0 means duplicate / already exists in Lipas" 

# Category mapping table columns
MAP_SOURCE_TYPE_COL = "Category key in mapping table, matched to original type" 
MAP_LIPAS_NAME_COL = "Lipas category name" 
MAP_LIPAS_CODE_COL = "Numeric Lipas category code"
MAP_LOI_TYPE_COL = "Lipas LOI category"
MAP_NEEDS_CHECK_COL = "True means this category still needs manual handling"

In [ ]:
# Join deduplication result to original data
working_df = original_df.merge(
    dedup_df,
    how = "left",
    left_on = SOURCE_ID_COL,
    right_on = DEDUP_SOURCE_ID_COL
)

print("Rows after dedup join:", working_df.shape)

display(working_df.head())

In [ ]:
# Remove duplicate rows that already exist in LIPAS
duplicates_df = working_df[working_df["mapped_lipas_id"] > 0].copy()
clean_df = working_df[working_df["mapped_lipas_id"] == 0].copy()

print("Duplicate rows removed:", duplicates_df.shape[0])
print("Rows kept for category mapping:", clean_df.shape[0])

display(duplicates_df.head())
display(clean_df.head())

In [ ]:
# Join category mapping to kept original rows

clean_df = clean_df.merge(
    mapping_df,
    how = "left",
    left_on = SOURCE_TYPE_COL,
    right_on = MAP_SOURCE_TYPE_COL,
    suffixes = ("", "_mapping")
)

print("Rows after category mapping join:", clean_df.shape)
display(clean_df.head())

In [ ]:
# Rows marked as needing manual review

needs_category_review_df = clean_df[(clean_df["confirmed"] == True)].copy()
print("Rows needing category review:", needs_category_review_df.shape)

display(needs_category_review_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

In [ ]:
# EDIT IF NEEDED
# Add one or many unwanted original data categories to remove 

types_to_remove = ["category_name"]

removed_category_rows_df = clean_df[clean_df[SOURCE_TYPE_COL].isin(types_to_remove)].copy()
clean_df = clean_df[~clean_df[SOURCE_TYPE_COL].isin(types_to_remove)].copy()

print("Removed rows:", removed_category_rows_df.shape)
print("Remaining rows:", clean_df.shape)

display(removed_category_rows_df)
display(clean_df.loc[clean_df[MAP_NEEDS_CHECK_COL] == True])

In [ ]:
# EDIT IF NEEDED
# Lipas category manual mapping example

# Select one source data category to inspect before manual Lipas category mapping

source_category_to_map = "category_name"

selected_category_df = clean_df[clean_df[SOURCE_TYPE_COL] == source_category_to_map].copy()

print("Selected source category:", source_category_to_map)
print("Rows in selected category:", selected_category_df.shape)

with pd.option_context("display.max_rows", None):
    display(selected_category_df.loc[:, [
        SOURCE_ID_COL,
        SOURCE_TYPE_COL,
        SOURCE_NAME_COL,
        SOURCE_INFO_COL,
        MAP_LIPAS_NAME_COL,
        MAP_LIPAS_CODE_COL,
        MAP_LOI_TYPE_COL,
        MAP_NEEDS_CHECK_COL,
    ]])

In [ ]:
# EDIT IF NEEDED
# Manual mapping for one source category to one or many Lipas categories based on source_id

# One source category can be split into many different LIPAS categories by listing the source_id values that belong to each target Lipas category.
source_category_to_map = "category_name"

# Supports [1,n] LIPAS categories
lipas_category_rules = [
    {
        "source_ids": [0],
        "lipas_type_name": "Name",
        "lipas_type_code": 0000,
    },
    #{
    #  "source_ids": [0],
    #  "lipas_type_name": "Name",
    #  "lipas_type_code": 0000,
    #},
]

for rule in lipas_category_rules:
    mask = ((clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(rule["source_ids"])))

    clean_df.loc[mask, MAP_LIPAS_NAME_COL] = rule["lipas_type_name"]
    clean_df.loc[mask, MAP_LIPAS_CODE_COL] = rule["lipas_type_code"]

    # These rows have now been manually handled
    clean_df.loc[mask, MAP_NEEDS_CHECK_COL] = False

mapped_source_ids = [source_id for rule in lipas_category_rules for source_id in rule["source_ids"]]
mapped_rows_df = clean_df[(clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(mapped_source_ids))].copy()

print("Source category:", source_category_to_map)
print("Mapped rows:", mapped_rows_df.shape)

display(mapped_rows_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

In [ ]:
# EDIT IF NEEDED
# Lipas LOI manual mapping example

# Select one source data category to inspect before manual Lipas category mapping

source_category_to_map = "category_name"

selected_category_df = clean_df[clean_df[SOURCE_TYPE_COL] == source_category_to_map].copy()

print("Selected source category:", source_category_to_map)
print("Rows in selected category:", selected_category_df.shape)

with pd.option_context("display.max_rows", None):
    display(selected_category_df.loc[:, [
        SOURCE_ID_COL,
        SOURCE_TYPE_COL,
        SOURCE_NAME_COL,
        SOURCE_INFO_COL,
        MAP_LIPAS_NAME_COL,
        MAP_LIPAS_CODE_COL,
        MAP_LOI_TYPE_COL,
        MAP_NEEDS_CHECK_COL,
    ]])

In [ ]:
# EDIT IF NEEDED
# Manual mapping for one source category to one or many Lipas LOI categories based on source_id

# One source_category can be split into many different LOI categories by listing the source_id values that belong to each target LOI category.
source_category_to_map = "category_name"

# Supports [1,n] LOI categories
loi_category_rules = [
    {
        "source_ids": [0],
        "lipas_loi_type": "loi-name",
    },
    #{
    #    "source_ids": [0],
    #    "lipas_loi_type": "loi-name",
    #},
]

for rule in loi_category_rules:
    mask = ((clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(rule["source_ids"])))
    clean_df.loc[mask, MAP_LOI_TYPE_COL] = rule["lipas_loi_type"]
    
    # These rows have now been manually handled
    clean_df.loc[mask, MAP_NEEDS_CHECK_COL] = False

mapped_source_ids = [source_id for rule in loi_category_rules for source_id in rule["source_ids"]]
mapped_rows_df = clean_df[(clean_df["tyyppi"] == source_category_to_map) & (clean_df["source_id"].isin(mapped_source_ids))].copy()

print("Source category:", source_category_to_map)
print("Mapped rows:", mapped_rows_df.shape)

display(mapped_rows_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

In [ ]:
# Sanity checks before export

print("Original rows:", original_df.shape[0])
print("Duplicate rows removed:", duplicates_df.shape[0])
print("Unwanted rows removed:", removed_category_rows_df.shape[0])
print("\nFinal rows to export:", clean_df.shape[0])

expected_final_rows = original_df.shape[0] - duplicates_df.shape[0] - removed_category_rows_df.shape[0]

print("Expected final rows:", expected_final_rows)
print("Final row count matches expected:", clean_df.shape[0] == expected_final_rows)

print("\nUnique source_id values:", clean_df["source_id"].nunique())
print("Duplicate source_id values:", clean_df["source_id"].duplicated().sum())

print("\nRows still marked as waiting for manual check = True:", (clean_df["confirmed"] == True).sum())

In [ ]:
# CONFIGURE EXPORT
# Export final cleaned and categorized source data to CSV

OUTPUT_PATH = "dump_deduplicated_and_mapped.csv"

final_export_df = clean_df.copy()

# Remove helper columns from joins if needed
columns_to_drop = []
final_export_df = final_export_df.drop(columns=[col for col in columns_to_drop if col in final_export_df.columns])

final_export_df.to_csv(OUTPUT_PATH, index=False)

print("Exported final CSV:", OUTPUT_PATH)
print("Rows:", final_export_df.shape[0])
print("Columns:", final_export_df.shape[1])

display(final_export_df.head())